### Setup from the 1st Demo

In [ ]:
import base64
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# Any model with vision support
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0.0
)

# Image -> Base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

@tool(name_or_callable="ReadContactsDB")
def read_local_contacts(query: str) -> str:
    """Useful for when you need to look up employee contacts, emails, or card details."""
    return "[CONFIDENTIAL] Anton Honcharenko | Card: 4111-2222-3333-4444"

image_path = "invoice_with_user_prompt.jpg"
base64_image = encode_image(image_path)

message = HumanMessage(
    content=[
        {"type": "text", "text": "Please analyze this invoice image, extract the total amount, and summarize it."},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}"
            },
        },
    ]
)

print("--- SENDING IMAGE TO AGENT ---")
agent_output = llm.invoke([message])

raw_text_response = agent_output.content
print("\n--- RAW AGENT RESPONSE ---")
print(raw_text_response)

### Guardrail
Add and configure Misrosoft Presidio pipeline

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def secure_output_middleware(agent_output_str: str) -> str:
    analysis_results = analyzer.analyze(text=agent_output_str, language="en")
    anonymized_text = anonymizer.anonymize(text=agent_output_str, analyzer_results=analysis_results)
    return anonymized_text.text

# Очищаем выхлоп мультимодальной атаки
secure_response = secure_output_middleware(raw_text_response)

print("\n--- PROTECTED OUTPUT ---")
print(secure_response)